In [ ]:
import snntorch as snn
from snntorch import spikeplot as splt
from snntorch import spikegen

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np
import itertools
height = 28
width = 28
batch_size = 128 # what does this effect
data_path = '/tmp/data/mnist'
dtype = torch.float

In [2]:
# def print_batch_accuracy(data, targets, train=False):
#     output, _ = net(data.view(batch_size, -1))
#     _, idx = output.sum(dim=0).max(1)
#     acc = np.mean((targets == idx).detach().cpu().numpy())

#     if train:
#         print(f"Train set accuracy for a single minibatch: {acc*100:.2f}%")
#     else:
#         print(f"Test set accuracy for a single minibatch: {acc*100:.2f}%")

# def train_printer():
#     print(f"Epoch {epoch}, Iteration {iter_counter}")
#     print(f"Train Set Loss: {loss_hist[counter]:.2f}")
#     print(f"Test Set Loss: {test_loss_hist[counter]:.2f}")
#     print_batch_accuracy(data, targets, train=True)
#     print_batch_accuracy(test_data, test_targets, train=False)
#     print("\n")

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
transform = transforms.Compose([
            transforms.Resize((height, width)),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize((0,), (1,))])
# what size was the data before?

mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)


In [ ]:
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
num_inputs = height * width
num_hidden = 1000
num_outputs = 10

# Temporal Dynamics
# experiment setting these to different values
time_steps = 25
beta = 0.95

# why layers and not individual neurons

In [ ]:
train_batch = iter(train_loader)
meta_counter = 0
for data, targets in train_batch:
    print(data.view(batch_size,-1))
    print(data.view(batch_size,-1).size())
    print(data.size())
    counter = 0
    # data is [128,1,28,28]
    # 128 28 by 28 images 
    # targets is a list of numeric labels
    print('target size', targets.size(), targets)
    for i in data:
        print(i.size())
        counter += 1
        # i[0] is image
        # i is [i[0]]
        # plt.figure()
        # plt.imshow(i[0])
        if counter > 5:
            break
    meta_counter += 1
    break


In [7]:
num_steps = time_steps
# Define Network
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Initialize layers
        self.fc1 = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta)
        self.fc2 = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x):

        # Initialize hidden states at t=0
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        # Record the final layer
        spk2_rec = []
        mem2_rec = []

        for step in range(num_steps):
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_rec.append(spk2)
            mem2_rec.append(mem2)

        return torch.stack(spk2_rec, dim=0), torch.stack(mem2_rec, dim=0)

# Load the network onto CUDA if available
net = Net().to(device)
loss = nn.CrossEntropyLoss()
temporal_loss = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=5e-4, betas=(0.9, 0.999))


# define training class

# training

In [8]:
import optuna

c:\Users\janso\easy_access_files\UCSC-CS-Classwork\cmpm118\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
def print_batch_accuracy(data, targets, train=False):
    output, _ = net(data.view(batch_size, -1))
    _, idx = output.sum(dim=0).max(1)
    acc = np.mean((targets == idx).detach().cpu().numpy())

    if train:
        print(f"Train set accuracy for a single minibatch: {acc*100:.2f}%")
    else:
        print(f"Test set accuracy for a single minibatch: {acc*100:.2f}%")

def train_printer():
    print(f"Epoch {epoch}, Iteration {iter_counter}")
    print(f"Train Set Loss: {loss_hist[counter]:.2f}")
    print(f"Test Set Loss: {test_loss_hist[counter]:.2f}")
    print_batch_accuracy(data, targets, train=True)
    print_batch_accuracy(test_data, test_targets, train=False)
    print("\n")

In [12]:
# def objective(trial):
static_threshold = 2 # trial.suggest_int('static_threshold', 2,13)
# suggest_float 6, 2
num_epochs = 2
num_steps = time_steps
loss_hist = []
test_loss_hist = []
counter = 0
temporal_threshold = 3
# Outer training loop
for epoch in range(num_epochs):
    iter_counter = 0
    train_batch = iter(train_loader)

    # Minibatch training loop
    for data, targets in train_batch:
        data = data.to(device)
        targets = targets.to(device)

        # forward pass
        net.train()
        spk_rec, mem_rec = net(data.view(batch_size, -1))

        accuracy_count = 0 # accuracy_count/len(spk_rec[step])
        spiked_arrs = 0
        # for image_idx in range(len(spk_rec[step])): # spk_rec[step] = 128 x 10 
        #     if 1 in spk_rec[step][image_idx]: # if any neuron spiked
        #         spiked_arrs+=1
        #     if spk_rec[step][image_idx][targets[image_idx]-1] == 1: # if the correct neuron spiked
        #         print(targets[image_idx]-1, spk_rec[step][image_idx])
        #         accuracy_count += 1
        # print("temporal threshold now", temporal_threshold, accuracy_count/spiked_arrs, accuracy_count, spiked_arrs)
        # if accuracy_count/spiked_arrs < .6:
        #     temporal_threshold += .2
        # if accuracy_count/spiked_arrs > .8:
        #     temporal_threshold -= .2

        # initialize the loss & sum over time
        loss_val = torch.zeros((1), dtype=dtype, device=device)
        for step in range(num_steps):
            if step < static_threshold:
                # 128 x 10 --> [1,0,0,0,0,0,0,0,0] [0,0,0,0,0,0,0,0,0]
                target_zeros = torch.zeros_like(spk_rec[step])
                loss_val += temporal_loss(spk_rec[step], target_zeros)
            loss_val += loss(mem_rec[step], targets)

        # Gradient calculation + weight update
        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()

        # Store loss history for future plotting
        loss_hist.append(loss_val.item())

        # Test set
        with torch.no_grad():
            net.eval()
            test_data, test_targets = next(iter(test_loader))
            test_data = test_data.to(device)
            test_targets = test_targets.to(device)

            # Test set forward pass
            test_spk, test_mem = net(test_data.view(batch_size, -1))

            # Test set loss
            test_loss = torch.zeros((1), dtype=dtype, device=device)
            for step in range(num_steps):
                test_loss += loss(test_mem[step], test_targets)
            test_loss_hist.append(test_loss.item())

            # Print train/test loss/accuracy
            if counter % 50 == 0:
                train_printer()
            counter += 1
            iter_counter +=1
# return loss_val.item()

Epoch 0, Iteration 0
Train Set Loss: 4.53
Test Set Loss: 4.34
Train set accuracy for a single minibatch: 96.88%
Test set accuracy for a single minibatch: 95.31%


Epoch 0, Iteration 50
Train Set Loss: 5.07
Test Set Loss: 3.68
Train set accuracy for a single minibatch: 98.44%
Test set accuracy for a single minibatch: 96.88%


Epoch 0, Iteration 100
Train Set Loss: 4.07
Test Set Loss: 2.17
Train set accuracy for a single minibatch: 96.88%
Test set accuracy for a single minibatch: 98.44%


Epoch 0, Iteration 150
Train Set Loss: 4.73
Test Set Loss: 3.01
Train set accuracy for a single minibatch: 96.88%
Test set accuracy for a single minibatch: 97.66%


Epoch 0, Iteration 200
Train Set Loss: 3.54
Test Set Loss: 3.12
Train set accuracy for a single minibatch: 98.44%
Test set accuracy for a single minibatch: 96.09%


Epoch 0, Iteration 250
Train Set Loss: 3.71
Test Set Loss: 4.12
Train set accuracy for a single minibatch: 97.66%
Test set accuracy for a single minibatch: 97.66%


Epoch 0, Iter

In [11]:
# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=2)
# print('best hyperparamters', study.best_params)